# Massive Activations in FLUX — Colab runner

Two experiments sharing one capture/ranking core:

- **Part 1 — Figure 3.** Top-/bottom-/random-k channel masks vs. a BiRefNet pseudo-GT, and the layer-wise mIoU curve (Fig 3D). Model is swappable — pick a preset in §5.
- **Part 2 — Channel stability.** For each (prompt, seed), at a fixed block + timestep, how much do the top channel *identities* change across generations? See `src/experiments/channel_stability.py`.

Sections 1–4 (setup, install, HF auth) are shared. Run Part 1 (§5–8) and/or Part 2 (§9–11) independently; §12 downloads results.

**Before you run:** `Runtime > Change runtime type` → GPU, and add an `HF_TOKEN` under Colab's 🔑 *Secrets* (needed for gated checkpoints like FLUX.1-dev — accept its license on the model page first).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader \
    || echo "No GPU detected — set Runtime > Change runtime type > GPU"
import sys

print(sys.version)

## 1. Setup

In [ ]:
import os

REPO_URL = "https://github.com/BrendanGho/massive-activations-fig3.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
REPO_DIR = "/content/massive-activations-fig3"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists — pulling latest {BRANCH}...")
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}
%cd {REPO_DIR}

!pip install -q -e ".[fig3]"
import torch

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    major, _minor = torch.cuda.get_device_capability(0)
    DEVICE = "cuda"
    # bf16 needs Ampere+ (compute capability >= 8, e.g. A100/L4); T4 (cc 7.5) falls back to fp16.
    DTYPE = "bf16" if major >= 8 else "fp16"
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
        "| compute capability:",
        major,
        "| using dtype:",
        DTYPE,
    )
else:
    DEVICE = "cpu"
    DTYPE = "fp32"
    print("No GPU — falling back to CPU/fp32. This will be extremely slow; use a GPU runtime.")

In [3]:
# mounting google drive (optional)
USE_DRIVE = True  # @param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/Research/MA"
else:
    DRIVE_ROOT = "/content/figure3_repro"
import os

os.makedirs(DRIVE_ROOT, exist_ok=True)
print("DRIVE_ROOT:", DRIVE_ROOT)

Mounted at /content/drive
DRIVE_ROOT: /content/drive/MyDrive/Research/MA


In [4]:
from huggingface_hub import login

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    login(token=hf_token)
    print("Logged in via Colab secret HF_TOKEN.")
else:
    login()

Logged in via Colab secret HF_TOKEN.


In [23]:
MODEL_PRESETS = {
    "flux2-klein": {
        "model_ckpt": "black-forest-labs/FLUX.2-klein-4B",
        "num_denoising_steps": 4,
        "guidance_scale": None,
        "offload_below_gb": 24,  # ~16GB in half precision
    },
    "flux-schnell": {
        "model_ckpt": "black-forest-labs/FLUX.1-schnell",
        "num_denoising_steps": 1,
        "guidance_scale": None,
        "offload_below_gb": 24,
    },
    "flux1-dev": {
        "model_ckpt": "black-forest-labs/FLUX.1-dev",  # gated — accept the license first
        "num_denoising_steps": 28,
        "guidance_scale": 3.5,
        "offload_below_gb": 40,  # ~24GB in half precision
    },
}


def model_dir(model_name: str) -> str:
    d = f"{DRIVE_ROOT}/{model_name}"
    os.makedirs(d, exist_ok=True)
    return d

# Part 1 — Figure 3

Reproduce the qualitative masks (Fig 3A–C) and the layer-wise mIoU curve (Fig 3D).


In [ ]:
### CONFIG BELOW
import yaml

ACTIVE_MODEL = "flux-schnell"  # @param ["flux2-klein", "flux-schnell", "flux1-dev"]
BIREFNET_WEIGHTS = "ZhengPeng7/BiRefNet"  # @param {type:"string"}
preset = MODEL_PRESETS[ACTIVE_MODEL]
MODEL_DIR = model_dir(ACTIVE_MODEL)
SMOKE_OUTPUT_DIR = f"{MODEL_DIR}/smoke/outputs"
SMOKE_CACHE_DIR = f"{MODEL_DIR}/smoke/cache"
os.makedirs(SMOKE_OUTPUT_DIR, exist_ok=True)
os.makedirs(SMOKE_CACHE_DIR, exist_ok=True)
PROMPT_SOURCE = f"{SMOKE_OUTPUT_DIR}/smoke_test_prompts.txt"
_SMOKE_PROMPTS = [
    "a red bicycle leaning against a brick wall",
    "a golden retriever sitting on a wooden dock at sunset",
    "a bowl of fresh strawberries on a marble countertop",
    "an astronaut planting a flag on the moon",
]
if not os.path.isfile(PROMPT_SOURCE):
    with open(PROMPT_SOURCE, "w") as fh:
        fh.write("\n".join(_SMOKE_PROMPTS) + "\n")
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    offload = total_gb < preset["offload_below_gb"]
    print(f"GPU memory: {total_gb:.1f} GB -> cpu offload: {offload}")
else:
    offload = False

config = {
    "model_ckpt": preset["model_ckpt"],
    "prompt_source": PROMPT_SOURCE,
    "birefnet_weights": BIREFNET_WEIGHTS,
    "output_dir": SMOKE_OUTPUT_DIR,
    "activation_cache_dir": SMOKE_CACHE_DIR,
    "num_denoising_steps": preset["num_denoising_steps"],
    "guidance_scale": preset["guidance_scale"],
    "resolution": 1024,
    "top_k": 12,
    "layers": "all",
    "random_k_trials": 5,
    "seed": 0,
    "device": DEVICE,
    "dtype": DTYPE,
    "offload": offload,
    "batch_size": 1,
    "num_example_prompts": 8,
    "cache_batch_size": 32,
}

CONFIG_PATH = "configs/colab.yaml"
with open(CONFIG_PATH, "w") as fh:
    yaml.safe_dump(config, fh, sort_keys=False)
print(open(CONFIG_PATH).read())

## 6. Quick smoke test

Runs 4 built-in prompts end to end; catches config, auth, or model-loading issues


In [ ]:
!python -m src.stage1_generate_and_cache --config {CONFIG_PATH} --fused --limit 4
!python -m src.stage4_evaluate_figure3d --config {CONFIG_PATH}

In [ ]:
import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f"{SMOKE_OUTPUT_DIR}/figure3d_results.csv")
display(df.pivot(index="layer", columns="strategy", values="mean_miou"))
display(Image(filename=f"{SMOKE_OUTPUT_DIR}/figure3d_curve.png"))

## 7. Full run (1,600 GenAI-Bench prompts)

Uses the bundled `data/genai_prompts.jsonl`


In [ ]:
FULL_OUTPUT_DIR = f"{MODEL_DIR}/full/outputs"
FULL_CACHE_DIR = f"{MODEL_DIR}/full/cache"
os.makedirs(FULL_OUTPUT_DIR, exist_ok=True)
os.makedirs(FULL_CACHE_DIR, exist_ok=True)

full_config = dict(config)  # reuse the model/dtype/offload knobs from section 5
full_config.update(
    prompt_source="data/genai_prompts.jsonl",
    output_dir=FULL_OUTPUT_DIR,
    activation_cache_dir=FULL_CACHE_DIR,
)
FULL_CONFIG_PATH = "configs/colab_full.yaml"
with open(FULL_CONFIG_PATH, "w") as fh:
    yaml.safe_dump(full_config, fh, sort_keys=False)
print(open(FULL_CONFIG_PATH).read())

!bash scripts/run_pipeline.sh {FULL_CONFIG_PATH}

In [ ]:
import glob

import pandas as pd
from IPython.display import Image, display

df = pd.read_csv(f"{FULL_OUTPUT_DIR}/figure3d_results.csv")
display(df.pivot(index="layer", columns="strategy", values="mean_miou"))
display(Image(filename=f"{FULL_OUTPUT_DIR}/figure3d_curve.png"))

for qdir in sorted(glob.glob(f"{FULL_OUTPUT_DIR}/qualitative/prompt_*"))[:3]:
    print(qdir)
    for png in sorted(glob.glob(f"{qdir}/*.png"))[:4]:
        display(Image(filename=png, width=220))

# Part 2 — Per-generation channel stability

*"Which channels are the largest for a given generation, and do they change depending on
what's being generated / from one generation to the next?"*

Fixes **one block** (`fixed_layer`, default 11 — the block the paper reports; earlier runs
used 24 ad hoc), runs a `prompts x seeds` scenario grid, and ranks channels per scenario by
`mean(abs(activations))` at the **last denoising step**. The grid is the design: same
prompt/different seed pairs measure generation-to-generation jitter, different-prompt pairs
measure content dependence. Two robustness checks ride along at no extra GPU cost: extra
activation snapshots at early/mid/last denoising steps (`capture_steps`) and a
token-localized secondary score (`secondary_metric: p999`). Requires §1–4 above.

In [ ]:
### CONFIG BELOW
import yaml

CS_ACTIVE_MODEL = "flux2-klein"  # @param ["flux2-klein", "flux-schnell", "flux1-dev"]
# Layer 11 matches configs/channel_stability.yaml (the block the paper reports works
# best); earlier runs used 24 ad hoc.
CS_FIXED_LAYER = 10  # @param {type:"integer"}
cs_preset = MODEL_PRESETS[CS_ACTIVE_MODEL]
CS_MODEL_DIR = model_dir(CS_ACTIVE_MODEL)  # same DRIVE_ROOT/<model> dir Part 1 writes to
CS_OUTPUT_DIR = f"{CS_MODEL_DIR}/channel_stability"
os.makedirs(CS_OUTPUT_DIR, exist_ok=True)
if torch.cuda.is_available():
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    CS_OFFLOAD = total_gb < cs_preset["offload_below_gb"]
    print(f"GPU memory: {total_gb:.1f} GB -> cpu offload: {CS_OFFLOAD}")
else:
    CS_OFFLOAD = True

# Early / mid / last denoising steps for the step-consistency check.
_n_steps = cs_preset["num_denoising_steps"]
CS_CAPTURE_STEPS = sorted({0, _n_steps // 2, _n_steps - 1})

cs_config = {
    "model_ckpt": cs_preset["model_ckpt"],
    "output_dir": CS_OUTPUT_DIR,
    "resolution": 1024,
    "num_denoising_steps": _n_steps,
    "guidance_scale": cs_preset["guidance_scale"],
    "device": DEVICE,
    "dtype": DTYPE,
    "offload": CS_OFFLOAD,
    "fixed_layer": CS_FIXED_LAYER,
    "capture_steps": CS_CAPTURE_STEPS,
    # scenarios = prompts x seeds. Same prompt/diff seed => generation-to-generation
    # jitter; different prompt => content dependence of the top channels.
    "prompts": [
        "a red bicycle leaning against a brick wall",
        "a golden retriever sitting on a wooden dock at sunset",
        "a bowl of fresh strawberries on a marble countertop",
        "an astronaut planting a flag on the moon",
    ],
    "seeds": [0, 1, 2],
    "top_n": 20,
    "agg_k": 10,
    "n_channel_maps": 5,
    "n_control_maps": 2,  # low-ranked control channel maps for representative scenarios
    "secondary_metric": "p999",  # token-localized complement to mean(abs)
}

CS_CONFIG_PATH = "configs/channel_stability_colab.yaml"
with open(CS_CONFIG_PATH, "w") as fh:
    yaml.safe_dump(cs_config, fh, sort_keys=False)

print(open(CS_CONFIG_PATH).read())
print(
    f"scenarios: {len(cs_config['prompts'])} prompts x {len(cs_config['seeds'])} seeds "
    f"= {len(cs_config['prompts']) * len(cs_config['seeds'])}"
)

In [ ]:
# run
!python -m src.experiments.channel_stability --config {CS_CONFIG_PATH}

## 11. Results


In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

# run() nests all outputs under a per-layer subfolder.
CS_RESULTS_DIR = f"{CS_OUTPUT_DIR}/layer_{CS_FIXED_LAYER}"

summary = json.load(open(f"{CS_RESULTS_DIR}/stability_summary.json"))
print(f"scenarios: {summary['n_scenarios']} | seeds: {summary['n_seeds']}")
for k, v in summary["per_k"].items():
    print(
        f"top-{k:>2}: {v['n_distinct_channels_used']:>3} distinct channels used | "
        f"Jaccard same-prompt/diff-seed={v['mean_jaccard_same_prompt_diff_seed']:.3f} "
        f"diff-prompt={v['mean_jaccard_diff_prompt']:.3f}"
    )

if "secondary_metric" in summary:
    sec = summary["secondary_metric"]
    agree = sec["mean_jaccard_primary_vs_secondary_per_k"]
    print(f"\nsecondary metric ({sec['name']}) agreement with mean(abs), per k:")
    print("  " + " | ".join(f"top-{k}: {v:.3f}" for k, v in agree.items()))

if "per_step_jaccard" in summary:
    print("\ntop-20 identity vs last step, per captured step:")
    for step, per_k in summary["per_step_jaccard"].items():
        print(f"  step {step:>3}: Jaccard(top-20) = {per_k['20']:.3f}")

if summary.get("figure_errors"):
    print("\nWARNING — figures that failed to render:")
    for name, tb in summary["figure_errors"].items():
        print(f"--- {name} ---\n{tb}")

print("\nfirst rows of the ordered top-20 table:")
display(pd.read_csv(f"{CS_RESULTS_DIR}/channel_stability_topk.csv").head(20))

# Headline figures: block-structured Jaccard + same-prompt vs diff-prompt pair plot;
# rank-colored scenario x channel heatmap; step consistency; qualitative contact sheet.
for fig in (
    "stability_overlap.png",
    "scenario_channel_heatmap.png",
    "step_consistency.png",
    "qualitative_summary.png",
):
    path = f"{CS_RESULTS_DIR}/{fig}"
    if os.path.isfile(path):
        display(Image(filename=path))
    else:
        print(f"(missing: {fig})")

print(f"\nper-scenario loose PNGs (for zooming): {CS_RESULTS_DIR}/scenarios/p<prompt>_s<seed>/")

In [55]:
from google.colab import runtime
runtime.unassign()

## 12. (Optional) Download outputs



In [ ]:
if not USE_DRIVE:
    import shutil

    from google.colab import files

    # MODEL_DIR (Part 1) and CS_MODEL_DIR (Part 2) are the same DRIVE_ROOT/<model> folder
    # when both parts used the same model, so dedupe by path before zipping.
    seen: dict[str, str] = {}
    if "MODEL_DIR" in globals():
        seen[MODEL_DIR] = ACTIVE_MODEL
    if "CS_MODEL_DIR" in globals():
        seen.setdefault(CS_MODEL_DIR, CS_ACTIVE_MODEL)
    for d, name in seen.items():
        if os.path.isdir(d) and os.listdir(d):
            archive = shutil.make_archive(f"/content/{name}", "zip", d)
            files.download(archive)
else:
    print("Outputs are already in Google Drive under:", DRIVE_ROOT)